# Cross-fitted TESSERA parent evidence for intercropping

This notebook tests a narrower and more operational question than the earlier three-class baseline: **can a field be detected as intercropped because its pixels carry evidence from both named monocrop parents through time?** It evaluates the raw field baseline, the interpretable parent-evidence representation alone, and a predeclared hybrid that combines both.

For each outer held-out-field fold, the parent axes are learned from field-balanced monocrop training pixels only. Intercropped fields never define the parent signatures. Pixel evidence is cross-fitted, summarized into distributional and spatial field features, and passed to a binary `intercrop vs monocrop` detector. Detector regularization and its operating threshold are selected inside the training data. The untouched outer fields are scored once.

The primary endpoint is `w1+w2+w3`. `w4` remains an out-of-contract sensitivity analysis. Potato-Maize is automatically exploratory while only 11 intercropped physical fields are available. Parent evidence is embedding affinity, not planted-area share, biomass, plant count, yield, or genetic ancestry.

Run all cells. The notebook saves every field-level prediction, every cross-fitted pixel parent score, all audit tables, seven PDF-ready figures, and one final `PDF_HANDOFF_V2` block.


In [ ]:
from collections import OrderedDict
from functools import reduce
import hashlib
import json
import os
from pathlib import Path
import tempfile

from IPython import get_ipython
from IPython.display import display

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pyproj import Transformer
from shapely import wkt as shapely_wkt
from shapely.ops import transform as shapely_transform
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


OUTPUT_DIR = Path(
    os.environ.get(
        "TESSERA_OUTPUT_DIR",
        "/mnt/noobjam/harvard_tessera_incremental_v2",
    )
).expanduser().resolve()
EXPORT_BASE = Path(
    os.environ.get(
        "TESSERA_SEPARABILITY_EXPORT_DIR",
        str(OUTPUT_DIR / "analysis" / "intercropping_temporal_separability_v1"),
    )
).expanduser().resolve()
WINDOWS = ("w1", "w2", "w3", "w4")
PRIMARY_WINDOWS = ("w1", "w2", "w3")
STAGES = OrderedDict(
    (
        ("w1", ("w1",)),
        ("w1+w2", ("w1", "w2")),
        ("w1+w2+w3", ("w1", "w2", "w3")),
        ("w1+w2+w3+w4", ("w1", "w2", "w3", "w4")),
    )
)
PRIMARY_STAGE = "w1+w2+w3"
LABELS = (
    "Maize",
    "Bean and Maize",
    "Irish Potato",
    "Bean",
    "Rice",
    "Irish Potato and Maize",
)
TASKS = OrderedDict(
    (
        (
            "bean_maize",
            {
                "title": "Bean - Maize family",
                "classes": ("Bean", "Maize", "Bean and Maize"),
                "mixture": "Bean and Maize",
            },
        ),
        (
            "potato_maize",
            {
                "title": "Irish Potato - Maize family",
                "classes": (
                    "Irish Potato",
                    "Maize",
                    "Irish Potato and Maize",
                ),
                "mixture": "Irish Potato and Maize",
            },
        ),
    )
)
CLASS_COLORS = {
    "Maize": "#D99B16",
    "Bean": "#25855A",
    "Irish Potato": "#7E57A6",
    "Bean and Maize": "#2A9D8F",
    "Irish Potato and Maize": "#B565A7",
    "Rice": "#3A78B4",
}
RANDOM_SEED = 20260709
MAX_OUTER_FOLDS = 5
INNER_FOLDS = 3
CANDIDATE_C = (0.01, 0.1, 1.0)
BOOTSTRAP_REPLICATES = 500
PIXEL_SIZE_M = 10.0

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "font.size": 10.5,
        "axes.titlesize": 12,
        "axes.labelsize": 10.5,
        "legend.fontsize": 8.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def _json(path):
    value = json.loads(Path(path).read_text())
    if not isinstance(value, dict):
        raise ValueError(f"expected JSON object in {path}")
    return value


def _canonical_sha(value):
    encoded = json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode()
    return hashlib.sha256(encoded).hexdigest()


def _task_key(row):
    return _canonical_sha(
        {
            "epsg": int(row.utm_epsg),
            "work_x": int(row.work_x_index),
            "work_y": int(row.work_y_index),
            "chunk_x": int(row.chunk_x_index),
            "chunk_y": int(row.chunk_y_index),
        }
    )[:24]


def _row_id(run_fingerprint, field_uid, pixel_id, window_id):
    text = f"{run_fingerprint}:{field_uid}:{pixel_id}:{window_id}"
    return hashlib.sha256(text.encode()).hexdigest()


def _finite_vector(value):
    if value is None:
        return None
    vector = np.asarray(value, dtype=np.float32)
    if vector.shape != (128,) or not np.isfinite(vector).all():
        return None
    norm = float(np.linalg.norm(vector))
    if norm <= 0:
        return None
    return (vector / norm).astype(np.float32)


def _atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    handle = tempfile.NamedTemporaryFile(
        prefix=f".{path.stem}.",
        suffix=".json.part",
        dir=path.parent,
        delete=False,
    )
    temporary = Path(handle.name)
    handle.close()
    try:
        temporary.write_text(json.dumps(value, indent=2, sort_keys=True, default=str) + "\n")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def _atomic_parquet(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".part")
    frame.to_parquet(temporary, index=False)
    os.replace(temporary, path)


def _save_figure(figure, export_root, filename):
    path = Path(export_root) / "figures" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.stem}.png.part")
    figure.savefig(temporary, format="png", dpi=200, bbox_inches="tight")
    os.replace(temporary, path)
    return path


def load_longitudinal_embeddings(output_dir):
    output_dir = Path(output_dir)
    required = ("run.json", "fields.parquet", "field_pixels.parquet")
    missing = [name for name in required if not (output_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(f"missing pipeline artifacts: {missing}")

    run = _json(output_dir / "run.json")
    run_fingerprint = str(run["run_fingerprint"])
    fields = pd.read_parquet(
        output_dir / "fields.parquet",
        columns=[
            "field_uid",
            "id",
            "landcover",
            "wkt",
            "utm_epsg",
            "pixel_count",
            "geometry_sha256",
        ],
    )
    membership_columns = [
        "field_uid",
        "landcover",
        "pixel_id",
        "utm_epsg",
        "pixel_x_index",
        "pixel_y_index",
        "pixel_longitude",
        "pixel_latitude",
        "work_x_index",
        "work_y_index",
        "chunk_x_index",
        "chunk_y_index",
    ]
    memberships = pd.read_parquet(
        output_dir / "field_pixels.parquet",
        columns=membership_columns,
    )
    fields["field_uid"] = fields["field_uid"].astype(str)
    fields["landcover"] = fields["landcover"].astype(str).str.strip()
    memberships["field_uid"] = memberships["field_uid"].astype(str)
    memberships["pixel_id"] = memberships["pixel_id"].astype(str)
    memberships["landcover"] = memberships["landcover"].astype(str).str.strip()

    fields["geometry_label_count"] = fields.groupby("geometry_sha256")[
        "landcover"
    ].transform("nunique")
    fields["analysis_unit_uid"] = fields.groupby(
        ["geometry_sha256", "landcover"],
        sort=False,
    )["field_uid"].transform("min")
    fields["source_replica_count"] = fields.groupby(
        ["geometry_sha256", "landcover"],
        sort=False,
    )["field_uid"].transform("size")
    units = fields[
        fields["landcover"].isin(LABELS)
        & fields["geometry_label_count"].eq(1)
        & fields["field_uid"].eq(fields["analysis_unit_uid"])
    ].copy()
    if units["field_uid"].duplicated().any():
        raise RuntimeError("canonical field identifiers are not unique")
    canonical_ids = set(units["field_uid"])
    canonical_memberships = memberships[
        memberships["field_uid"].isin(canonical_ids)
    ].copy()
    if canonical_memberships.duplicated(["field_uid", "pixel_id"]).any():
        raise RuntimeError("canonical field memberships are duplicated")

    task_columns = [
        "utm_epsg",
        "work_x_index",
        "work_y_index",
        "chunk_x_index",
        "chunk_y_index",
    ]
    task_table = canonical_memberships[task_columns].drop_duplicates().copy()
    task_table["task_key"] = [
        _task_key(row) for row in task_table.itertuples(index=False)
    ]
    canonical_memberships = canonical_memberships.merge(
        task_table,
        on=task_columns,
        how="left",
        validate="many_to_one",
    )
    expected_counts = canonical_memberships.groupby("field_uid").size()
    used_files = set()
    loaded_by_window = {}

    for window in WINDOWS:
        window_parts = []
        for task_key, rows in canonical_memberships.groupby("task_key", sort=True):
            path = output_dir / "embeddings" / f"window_id={window}" / f"{task_key}.parquet"
            if not path.is_file():
                continue
            row_ids = [
                _row_id(run_fingerprint, field_uid, pixel_id, window)
                for field_uid, pixel_id in rows[
                    ["field_uid", "pixel_id"]
                ].itertuples(index=False, name=None)
            ]
            table = pq.read_table(
                path,
                columns=["row_id", "field_uid", "pixel_id", "outcome", "embedding"],
                filters=[("row_id", "in", row_ids)],
                partitioning=None,
            ).to_pandas()
            if not table.empty:
                window_parts.append(table)
                used_files.add(path)
        loaded = (
            pd.concat(window_parts, ignore_index=True)
            if window_parts
            else pd.DataFrame(
                columns=["row_id", "field_uid", "pixel_id", "outcome", "embedding"]
            )
        )
        loaded["field_uid"] = loaded["field_uid"].astype(str)
        loaded["pixel_id"] = loaded["pixel_id"].astype(str)
        if loaded.duplicated(["field_uid", "pixel_id"]).any():
            raise RuntimeError(f"duplicate field/pixel rows in {window}")
        loaded["_vector"] = loaded["embedding"].map(_finite_vector)
        loaded_by_window[window] = loaded.drop(columns="embedding")

    published_by_window = {}
    for window, loaded in loaded_by_window.items():
        counts = loaded.groupby("field_uid").size().reindex(expected_counts.index, fill_value=0)
        published_by_window[window] = set(counts[counts.eq(expected_counts)].index)
    common_published = reduce(set.intersection, published_by_window.values())

    pixel_field_count = canonical_memberships.groupby("pixel_id")["field_uid"].nunique()
    private_pixels = set(pixel_field_count[pixel_field_count.eq(1)].index)
    model_memberships = canonical_memberships[
        canonical_memberships["field_uid"].isin(common_published)
        & canonical_memberships["pixel_id"].isin(private_pixels)
    ].copy()
    allowed_pairs = set(
        model_memberships[["field_uid", "pixel_id"]].itertuples(index=False, name=None)
    )
    complete_sets = []
    for window, loaded in loaded_by_window.items():
        complete = loaded[
            loaded["outcome"].eq("complete") & loaded["_vector"].notna()
        ]
        complete_sets.append(
            set(complete[["field_uid", "pixel_id"]].itertuples(index=False, name=None))
            & allowed_pairs
        )
    common_pairs = reduce(set.intersection, complete_sets)
    scoreable_fields = {field_uid for field_uid, _ in common_pairs}
    if not scoreable_fields:
        raise RuntimeError("no complete private field pixels are shared by all windows")

    model_memberships = model_memberships[
        model_memberships["field_uid"].isin(scoreable_fields)
    ].drop(columns="task_key")
    timeline_parts = []
    for window, loaded in loaded_by_window.items():
        selected = loaded[
            loaded[["field_uid", "pixel_id"]]
            .apply(tuple, axis=1)
            .isin(common_pairs)
        ].drop(columns=["row_id", "outcome"])
        selected = selected.merge(
            model_memberships,
            on=["field_uid", "pixel_id"],
            how="left",
            validate="one_to_one",
        )
        selected["window_id"] = window
        timeline_parts.append(selected)
    timeline = pd.concat(timeline_parts, ignore_index=True)
    if timeline.duplicated(["field_uid", "pixel_id", "window_id"]).any():
        raise RuntimeError("longitudinal timeline contains duplicate rows")
    pair_counts = timeline.groupby(["field_uid", "pixel_id"])["window_id"].nunique()
    if not pair_counts.eq(len(WINDOWS)).all():
        raise RuntimeError("the exact pixel cohort is not present in every window")

    units = units[units["field_uid"].isin(scoreable_fields)].copy()
    field_labels = units.set_index("field_uid")["landcover"]
    timeline["landcover"] = timeline["field_uid"].map(field_labels)
    if timeline["landcover"].isna().any():
        raise RuntimeError("a timeline field lost its label")

    specs = {
        str(spec["window_id"]): dict(spec)
        for spec in run.get("config", {}).get("windows", [])
        if isinstance(spec, dict) and "window_id" in spec
    }
    window_records = []
    for window in WINDOWS:
        spec = specs.get(window, {"window_id": window})
        start = pd.Timestamp(str(spec.get("start", "2024-09-01")))
        end = pd.Timestamp(str(spec.get("end_exclusive", "2026-01-01")))
        duration = int((end - start).days)
        window_records.append(
            {
                "window_id": window,
                "start": start.date().isoformat(),
                "end_exclusive": end.date().isoformat(),
                "duration_days": duration,
                "contract_status": "in contract" if duration <= 366 else "OOD sensitivity",
            }
        )
    window_table = pd.DataFrame(window_records)

    file_records = [
        {
            "path": str(path.relative_to(output_dir)),
            "bytes": path.stat().st_size,
            "mtime_ns": path.stat().st_mtime_ns,
        }
        for path in sorted(used_files)
    ]
    snapshot_id = _canonical_sha(
        {
            "run_fingerprint": run_fingerprint,
            "files": file_records,
            "common_pairs": len(common_pairs),
        }
    )[:16]
    diagnostics = {
        "run_fingerprint": run_fingerprint,
        "snapshot_id": snapshot_id,
        "pipeline_complete_at_snapshot": (output_dir / "COMPLETED.json").is_file(),
        "source_field_rows": int(len(fields)),
        "canonical_supported_units": int(len(fields[
            fields["landcover"].isin(LABELS)
            & fields["geometry_label_count"].eq(1)
            & fields["field_uid"].eq(fields["analysis_unit_uid"])
        ])),
        "scoreable_physical_fields": int(len(units)),
        "scoreable_physical_pixels": int(len(common_pairs)),
        "shared_physical_pixels_removed": int(len(pixel_field_count) - len(private_pixels)),
    }
    return {
        "run": run,
        "fields": units.reset_index(drop=True),
        "timeline": timeline.reset_index(drop=True),
        "window_table": window_table,
        "diagnostics": diagnostics,
        "snapshot_id": snapshot_id,
    }


def build_field_features(timeline, fields):
    field_ids = sorted(fields["field_uid"].astype(str))
    field_table = fields.set_index("field_uid").loc[field_ids].reset_index()
    summaries = []
    mean_by_window = {}
    std_by_window = {}
    for window in WINDOWS:
        rows = timeline[timeline["window_id"].eq(window)]
        means = []
        standard_deviations = []
        for field_uid in field_ids:
            vectors = np.stack(
                rows.loc[rows["field_uid"].eq(field_uid), "_vector"].to_numpy()
            ).astype(np.float64)
            mean = vectors.mean(axis=0)
            standard_deviation = vectors.std(axis=0)
            means.append(mean)
            standard_deviations.append(standard_deviation)
            summaries.append(
                {
                    "field_uid": field_uid,
                    "window_id": window,
                    "landcover": str(field_table.loc[
                        field_table["field_uid"].eq(field_uid), "landcover"
                    ].iloc[0]),
                    "pixel_count": int(len(vectors)),
                    "dispersion": float(
                        np.linalg.norm(vectors - mean[None, :], axis=1).mean()
                    ),
                }
            )
        mean_by_window[window] = np.vstack(means)
        std_by_window[window] = np.vstack(standard_deviations)
    features = {
        stage: np.hstack(
            [
                component
                for window in stage_windows
                for component in (mean_by_window[window], std_by_window[window])
            ]
        )
        for stage, stage_windows in STAGES.items()
    }
    return {
        "field_table": field_table,
        "features": features,
        "mean_by_window": mean_by_window,
        "summary": pd.DataFrame(summaries),
    }


from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_recall_curve, precision_score


PARENT_EVIDENCE_EXPORT_BASE = Path(
    os.environ.get(
        "TESSERA_PARENT_EVIDENCE_EXPORT_DIR",
        str(OUTPUT_DIR / "analysis" / "intercropping_parent_evidence_v2"),
    )
).expanduser().resolve()
TARGET_TRAIN_FPR = 0.10
MAX_HELD_OUT_FPR = 0.15
MIN_CONFIRMATORY_MIXTURE_FIELDS = 30
PARENT_CROSSFIT_FOLDS = 4
PARENT_AXIS_C = 0.1
DETECTOR_C_GRID = (0.01, 0.1, 1.0, 10.0)
V2_BOOTSTRAP_REPLICATES = 500

PARENT_FEATURE_NAMES = (
    "mean_parent_a",
    "std_parent_a",
    "q10_parent_a",
    "q25_parent_a",
    "median_parent_a",
    "q75_parent_a",
    "q90_parent_a",
    "iqr_parent_a",
    "q90_q10_parent_a",
    "parent_balance",
    "parent_a_tail_mass",
    "parent_b_tail_mass",
    "dual_parent_tail_mass",
    "intermediate_mass",
    "mean_parent_entropy",
    "spatial_parent_contrast",
    "spatial_parent_crossing",
)

MODEL_LABELS = OrderedDict(
    (
        ("raw_baseline", "Raw mean + std baseline"),
        ("parent_evidence", "Parent evidence only"),
        ("hybrid_parent_evidence", "Raw + parent evidence hybrid"),
    )
)
MODEL_COLORS = {
    "raw_baseline": "#8A8F98",
    "parent_evidence": "#156C78",
    "hybrid_parent_evidence": "#C45A2C",
}


def _field_positions(field_bundle):
    field_table = field_bundle["field_table"]
    return {
        field_uid: position
        for position, field_uid in enumerate(field_table["field_uid"].astype(str))
    }


def _fit_parent_axis(field_bundle, timeline, positions, window, task):
    field_table = field_bundle["field_table"]
    labels = field_table["landcover"].astype(str).to_numpy()
    parent_a, parent_b = task["classes"][:2]
    positions = np.asarray(positions, dtype=np.int64)
    selected = positions[np.isin(labels[positions], (parent_a, parent_b))]
    selected_labels = labels[selected]
    if set(selected_labels) != {parent_a, parent_b}:
        raise RuntimeError("a parent axis is missing one monocrop class")
    if np.any(selected_labels == task["mixture"]):
        raise RuntimeError("an intercropped field entered a parent-axis fit")

    training_field_ids = field_table.loc[selected, "field_uid"].astype(str)
    rows = timeline[
        timeline["window_id"].eq(window)
        & timeline["field_uid"].astype(str).isin(training_field_ids)
    ].sort_values(["field_uid", "pixel_id"], kind="stable")
    if set(rows["landcover"].astype(str)) != {parent_a, parent_b}:
        raise RuntimeError("pixel parent-axis rows contain unexpected labels")
    vectors = np.stack(rows["_vector"].to_numpy()).astype(np.float64)
    field_pixel_count = rows.groupby("field_uid").size()
    class_field_count = rows.groupby("landcover")["field_uid"].nunique()
    weights = np.asarray(
        [
            1.0 / (class_field_count[label] * field_pixel_count[field_uid])
            for field_uid, label in rows[
                ["field_uid", "landcover"]
            ].itertuples(index=False, name=None)
        ],
        dtype=np.float64,
    )
    weights *= len(weights) / weights.sum()
    scaler = StandardScaler().fit(vectors, sample_weight=weights)
    model = LogisticRegression(
        C=PARENT_AXIS_C,
        max_iter=5000,
        solver="lbfgs",
        random_state=RANDOM_SEED,
    )
    model.fit(
        scaler.transform(vectors),
        rows["landcover"].astype(str),
        sample_weight=weights,
    )
    return {
        "scaler": scaler,
        "model": model,
        "parent_a": parent_a,
        "parent_b": parent_b,
        "training_fields": int(len(selected)),
        "training_pixels": int(len(rows)),
    }


def _score_parent_axis(axis, timeline, window, field_ids):
    field_ids = set(map(str, field_ids))
    rows = timeline[
        timeline["window_id"].eq(window)
        & timeline["field_uid"].astype(str).isin(field_ids)
    ].sort_values(["field_uid", "pixel_id"], kind="stable")
    if rows.empty:
        raise RuntimeError(f"no pixels to score for {window}")
    vectors = np.stack(rows["_vector"].to_numpy()).astype(np.float64)
    probability = axis["model"].predict_proba(
        axis["scaler"].transform(vectors)
    )
    class_index = {
        label: index for index, label in enumerate(axis["model"].classes_)
    }
    result = rows[
        [
            "field_uid",
            "pixel_id",
            "landcover",
            "utm_epsg",
            "pixel_x_index",
            "pixel_y_index",
        ]
    ].copy()
    result["parent_a_probability"] = probability[
        :, class_index[axis["parent_a"]]
    ]
    return result.reset_index(drop=True)


def _cross_fitted_parent_scores(
    field_bundle,
    timeline,
    outer_train_positions,
    outer_test_positions,
    task,
    task_seed,
):
    field_table = field_bundle["field_table"]
    labels = field_table["landcover"].astype(str).to_numpy()
    field_ids = field_table["field_uid"].astype(str).to_numpy()
    outer_train_positions = np.asarray(outer_train_positions, dtype=np.int64)
    outer_test_positions = np.asarray(outer_test_positions, dtype=np.int64)
    train_labels = labels[outer_train_positions]
    counts = pd.Series(train_labels).value_counts().reindex(task["classes"])
    fold_count = int(min(PARENT_CROSSFIT_FOLDS, counts.min()))
    if fold_count < 2:
        raise RuntimeError("parent-evidence cross-fitting needs at least two folds")

    splitter = StratifiedKFold(
        n_splits=fold_count,
        shuffle=True,
        random_state=task_seed,
    )
    splits = list(splitter.split(np.zeros(len(train_labels)), train_labels))
    train_by_window = {window: [] for window in WINDOWS}
    test_by_window = {window: [] for window in WINDOWS}
    audit_records = []

    for base_fold, (fit_local, hold_local) in enumerate(splits):
        fit_positions = outer_train_positions[fit_local]
        hold_positions = outer_train_positions[hold_local]
        fit_ids = field_ids[fit_positions]
        hold_ids = field_ids[hold_positions]
        test_ids = field_ids[outer_test_positions]
        if set(fit_ids) & set(hold_ids):
            raise RuntimeError("parent-axis fit and holdout fields overlap")
        if set(fit_ids) & set(test_ids):
            raise RuntimeError("outer test fields entered a parent-axis fit")

        for window in WINDOWS:
            axis = _fit_parent_axis(
                field_bundle, timeline, fit_positions, window, task
            )
            held = _score_parent_axis(axis, timeline, window, hold_ids)
            held["base_fold"] = base_fold
            train_by_window[window].append(held)
            test_by_window[window].append(
                _score_parent_axis(axis, timeline, window, test_ids)
            )
            audit_records.append(
                {
                    "base_fold": base_fold,
                    "window_id": window,
                    "parent_axis_training_fields": axis["training_fields"],
                    "parent_axis_training_pixels": axis["training_pixels"],
                    "held_fields": len(hold_ids),
                    "outer_test_fields": len(test_ids),
                    "mixture_fields_in_parent_fit": 0,
                }
            )

    cross_fitted_train = {}
    ensemble_test = {}
    for window in WINDOWS:
        train_rows = pd.concat(train_by_window[window], ignore_index=True).sort_values(
            ["field_uid", "pixel_id"], kind="stable"
        )
        if train_rows.duplicated(["field_uid", "pixel_id"]).any():
            raise RuntimeError("a training pixel received multiple cross-fitted scores")
        expected_train_fields = set(field_ids[outer_train_positions])
        if set(train_rows["field_uid"].astype(str)) != expected_train_fields:
            raise RuntimeError("cross-fitted parent scores lost training fields")
        cross_fitted_train[window] = train_rows.reset_index(drop=True)

        ordered = [
            frame.sort_values(["field_uid", "pixel_id"], kind="stable").reset_index(
                drop=True
            )
            for frame in test_by_window[window]
        ]
        keys = ordered[0][["field_uid", "pixel_id"]]
        for frame in ordered[1:]:
            if not frame[["field_uid", "pixel_id"]].equals(keys):
                raise RuntimeError("test pixel alignment changed across parent axes")
        averaged = ordered[0].copy()
        averaged["parent_a_probability"] = np.mean(
            [frame["parent_a_probability"].to_numpy() for frame in ordered],
            axis=0,
        )
        ensemble_test[window] = averaged

    return cross_fitted_train, ensemble_test, pd.DataFrame(audit_records)


def _spatial_parent_features(rows):
    probability = {
        (int(row.utm_epsg), int(row.pixel_x_index), int(row.pixel_y_index)): float(
            row.parent_a_probability
        )
        for row in rows.itertuples(index=False)
    }
    differences = []
    crossings = []
    for (epsg, x_index, y_index), value in probability.items():
        for neighbor in (
            (epsg, x_index + 1, y_index),
            (epsg, x_index, y_index + 1),
        ):
            if neighbor not in probability:
                continue
            neighbor_value = probability[neighbor]
            differences.append(abs(value - neighbor_value))
            crossings.append((value >= 0.5) != (neighbor_value >= 0.5))
    if not differences:
        return np.nan, np.nan
    return float(np.mean(differences)), float(np.mean(crossings))


def _summarize_parent_scores(frame, ordered_field_ids):
    records = []
    for field_uid in map(str, ordered_field_ids):
        rows = frame[frame["field_uid"].astype(str).eq(field_uid)]
        if rows.empty:
            raise RuntimeError(f"field {field_uid} has no parent-evidence pixels")
        values = np.clip(
            rows["parent_a_probability"].to_numpy(dtype=np.float64),
            1e-6,
            1.0 - 1e-6,
        )
        q10, q25, median, q75, q90 = np.quantile(
            values, [0.10, 0.25, 0.50, 0.75, 0.90]
        )
        parent_a_tail = float(np.mean(values >= 0.80))
        parent_b_tail = float(np.mean(values <= 0.20))
        spatial_contrast, spatial_crossing = _spatial_parent_features(rows)
        entropy = -values * np.log(values) - (1.0 - values) * np.log(
            1.0 - values
        )
        records.append(
            {
                "field_uid": field_uid,
                "landcover": str(rows["landcover"].iloc[0]),
                "mean_parent_a": float(values.mean()),
                "std_parent_a": float(values.std()),
                "q10_parent_a": float(q10),
                "q25_parent_a": float(q25),
                "median_parent_a": float(median),
                "q75_parent_a": float(q75),
                "q90_parent_a": float(q90),
                "iqr_parent_a": float(q75 - q25),
                "q90_q10_parent_a": float(q90 - q10),
                "parent_balance": float(1.0 - abs(2.0 * values.mean() - 1.0)),
                "parent_a_tail_mass": parent_a_tail,
                "parent_b_tail_mass": parent_b_tail,
                "dual_parent_tail_mass": float(
                    2.0 * min(parent_a_tail, parent_b_tail)
                ),
                "intermediate_mass": float(
                    np.mean((values > 0.20) & (values < 0.80))
                ),
                "mean_parent_entropy": float(entropy.mean()),
                "spatial_parent_contrast": spatial_contrast,
                "spatial_parent_crossing": spatial_crossing,
            }
        )
    result = pd.DataFrame(records).set_index("field_uid")
    return result.loc[list(map(str, ordered_field_ids))]


def _progressive_parent_matrix(summary_by_window, stage_windows, field_ids):
    field_ids = list(map(str, field_ids))
    matrices = []
    names = []
    window_values = {}
    for window in stage_windows:
        values = summary_by_window[window].loc[field_ids, PARENT_FEATURE_NAMES].to_numpy(
            dtype=np.float64
        )
        window_values[window] = values
        matrices.append(values)
        names.extend(f"{window}:{name}" for name in PARENT_FEATURE_NAMES)
    for previous, current in zip(stage_windows[:-1], stage_windows[1:], strict=True):
        matrices.append(window_values[current] - window_values[previous])
        names.extend(
            f"delta_{current}-{previous}:{name}" for name in PARENT_FEATURE_NAMES
        )
    return np.hstack(matrices), names


def _detector(c_value):
    return Pipeline(
        [
            (
                "impute",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            ("scale", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    C=float(c_value),
                    class_weight="balanced",
                    max_iter=5000,
                    solver="liblinear",
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )


def _operating_threshold(y_true, scores, target_fpr=TARGET_TRAIN_FPR):
    y_true = np.asarray(y_true, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float64)
    thresholds = np.r_[
        np.nextafter(scores.max(), np.inf),
        np.unique(scores)[::-1],
    ]
    candidates = []
    for threshold in thresholds:
        predicted = scores >= threshold
        negatives = y_true == 0
        positives = y_true == 1
        fpr = float(np.mean(predicted[negatives]))
        recall = float(np.mean(predicted[positives]))
        precision = float(
            precision_score(y_true, predicted, zero_division=0)
        )
        if fpr <= target_fpr + 1e-12:
            candidates.append((recall, precision, -fpr, float(threshold)))
    if not candidates:
        raise RuntimeError("no threshold satisfies the target false-positive rate")
    recall, precision, negative_fpr, threshold = max(candidates)
    return {
        "threshold": threshold,
        "training_recall": recall,
        "training_precision": precision,
        "training_fpr": -negative_fpr,
    }


def _select_and_fit_detector(x_train, y_train, x_test, seed):
    y_train = np.asarray(y_train, dtype=np.int8)
    positive_count = int(y_train.sum())
    negative_count = int(len(y_train) - positive_count)
    fold_count = int(min(INNER_FOLDS, positive_count, negative_count))
    if fold_count < 2:
        raise RuntimeError("detector selection needs at least two inner folds")
    splitter = StratifiedKFold(
        n_splits=fold_count,
        shuffle=True,
        random_state=seed,
    )
    splits = list(splitter.split(np.zeros(len(y_train)), y_train))
    candidates = []
    for c_value in DETECTOR_C_GRID:
        oof_scores = np.full(len(y_train), np.nan, dtype=np.float64)
        for fit, validation in splits:
            model = _detector(c_value)
            model.fit(x_train[fit], y_train[fit])
            oof_scores[validation] = model.predict_proba(x_train[validation])[:, 1]
        if not np.isfinite(oof_scores).all():
            raise RuntimeError("inner detector predictions are incomplete")
        candidates.append(
            {
                "c_value": float(c_value),
                "average_precision": float(
                    average_precision_score(y_train, oof_scores)
                ),
                "auroc": float(roc_auc_score(y_train, oof_scores)),
                "scores": oof_scores,
            }
        )
    selected = max(
        candidates,
        key=lambda row: (row["average_precision"], -row["c_value"]),
    )
    operating = _operating_threshold(y_train, selected["scores"])
    model = _detector(selected["c_value"])
    model.fit(x_train, y_train)
    test_scores = model.predict_proba(x_test)[:, 1]
    return {
        "scores": test_scores,
        "predicted": test_scores >= operating["threshold"],
        "c_value": selected["c_value"],
        "inner_average_precision": selected["average_precision"],
        "inner_auroc": selected["auroc"],
        **operating,
    }


def _binary_metrics(y_true, scores, predicted):
    y_true = np.asarray(y_true, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=bool)
    positives = y_true == 1
    negatives = y_true == 0
    prevalence = float(positives.mean())
    average_precision = float(average_precision_score(y_true, scores))
    recall = float(np.mean(predicted[positives]))
    fpr = float(np.mean(predicted[negatives]))
    precision = float(precision_score(y_true, predicted, zero_division=0))
    return {
        "auroc": float(roc_auc_score(y_true, scores)),
        "average_precision": average_precision,
        "average_precision_lift": float(average_precision / prevalence),
        "recall": recall,
        "specificity": 1.0 - fpr,
        "false_positive_rate": fpr,
        "precision": precision,
        "balanced_accuracy": 0.5 * (recall + 1.0 - fpr),
    }


def _stratified_bootstrap_indices(labels, rng):
    labels = np.asarray(labels, dtype=object)
    return np.concatenate(
        [
            rng.choice(positions, size=len(positions), replace=True)
            for positions in (
                np.flatnonzero(labels == label) for label in np.unique(labels)
            )
        ]
    )


def _performance_table(predictions):
    metric_names = tuple(
        _binary_metrics(
            np.array([0, 1]),
            np.array([0.1, 0.9]),
            np.array([False, True]),
        )
    )
    records = []
    grouped = predictions.groupby(["task_key", "model_key", "stage"], sort=False)
    for group_offset, ((task_key, model_key, stage), rows) in enumerate(grouped):
        rows = rows.reset_index(drop=True)
        metrics = _binary_metrics(
            rows["is_intercrop"], rows["intercrop_score"], rows["predicted_intercrop"]
        )
        rng = np.random.default_rng(RANDOM_SEED + 10000 + group_offset)
        bootstrap = {name: [] for name in metric_names}
        for _ in range(V2_BOOTSTRAP_REPLICATES):
            sample = _stratified_bootstrap_indices(rows["landcover"], rng)
            values = _binary_metrics(
                rows.loc[sample, "is_intercrop"],
                rows.loc[sample, "intercrop_score"],
                rows.loc[sample, "predicted_intercrop"],
            )
            for name, value in values.items():
                bootstrap[name].append(value)
        record = {
            "task_key": task_key,
            "task_title": TASKS[task_key]["title"],
            "model_key": model_key,
            "model": MODEL_LABELS[model_key],
            "stage": stage,
            "contract_status": (
                "primary" if stage != "w1+w2+w3+w4" else "OOD sensitivity"
            ),
            "fields": int(len(rows)),
            "mixture_fields": int(rows["is_intercrop"].sum()),
            "prevalence": float(rows["is_intercrop"].mean()),
            "median_selected_c": float(rows["selected_c"].median()),
            "median_threshold": float(rows["threshold"].median()),
        }
        for name, value in metrics.items():
            low, high = np.quantile(bootstrap[name], [0.025, 0.975])
            record[name] = value
            record[name + "_ci_low"] = float(low)
            record[name + "_ci_high"] = float(high)
        records.append(record)
    return pd.DataFrame(records)


def _paired_model_comparison(predictions):
    records = []
    for task_key in TASKS:
        for stage in STAGES:
            rows = predictions[
                predictions["task_key"].eq(task_key)
                & predictions["stage"].eq(stage)
            ]
            baseline = rows[rows["model_key"].eq("raw_baseline")].sort_values(
                "field_uid"
            )
            baseline_score = baseline["intercrop_score"].to_numpy()
            for candidate_offset, candidate_key in enumerate(
                ("parent_evidence", "hybrid_parent_evidence")
            ):
                candidate = rows[
                    rows["model_key"].eq(candidate_key)
                ].sort_values("field_uid")
                if not candidate["field_uid"].reset_index(drop=True).equals(
                    baseline["field_uid"].reset_index(drop=True)
                ):
                    raise RuntimeError("paired model predictions are misaligned")
                y = candidate["is_intercrop"].to_numpy(dtype=np.int8)
                labels = candidate["landcover"].to_numpy(dtype=object)
                candidate_score = candidate["intercrop_score"].to_numpy()
                observed = {
                    "delta_auroc": roc_auc_score(y, candidate_score)
                    - roc_auc_score(y, baseline_score),
                    "delta_average_precision": average_precision_score(
                        y, candidate_score
                    )
                    - average_precision_score(y, baseline_score),
                }
                rng = np.random.default_rng(
                    RANDOM_SEED
                    + 20000
                    + 1000 * list(TASKS).index(task_key)
                    + 10 * list(STAGES).index(stage)
                    + candidate_offset
                )
                bootstrap = {name: [] for name in observed}
                for _ in range(V2_BOOTSTRAP_REPLICATES):
                    sample = _stratified_bootstrap_indices(labels, rng)
                    sample_y = y[sample]
                    bootstrap["delta_auroc"].append(
                        roc_auc_score(sample_y, candidate_score[sample])
                        - roc_auc_score(sample_y, baseline_score[sample])
                    )
                    bootstrap["delta_average_precision"].append(
                        average_precision_score(sample_y, candidate_score[sample])
                        - average_precision_score(sample_y, baseline_score[sample])
                    )
                record = {
                    "task_key": task_key,
                    "stage": stage,
                    "candidate_model_key": candidate_key,
                    **observed,
                }
                for name, values in bootstrap.items():
                    low, high = np.quantile(values, [0.025, 0.975])
                    record[name + "_ci_low"] = float(low)
                    record[name + "_ci_high"] = float(high)
                records.append(record)
    return pd.DataFrame(records)


def _parent_axis_diagnostics(field_evidence):
    records = []
    for task_offset, (task_key, task) in enumerate(TASKS.items()):
        parent_a, parent_b = task["classes"][:2]
        for window_offset, window in enumerate(WINDOWS):
            rows = field_evidence[
                field_evidence["task_key"].eq(task_key)
                & field_evidence["window_id"].eq(window)
                & field_evidence["landcover"].isin((parent_a, parent_b))
            ].reset_index(drop=True)
            y = rows["landcover"].eq(parent_a).to_numpy(dtype=np.int8)
            score = rows["mean_parent_a"].to_numpy(dtype=np.float64)
            predicted = score >= 0.5
            auroc = float(roc_auc_score(y, score))
            balanced = float(balanced_accuracy_score(y, predicted))
            rng = np.random.default_rng(
                RANDOM_SEED + 30000 + 100 * task_offset + window_offset
            )
            auroc_bootstrap = []
            balanced_bootstrap = []
            for _ in range(V2_BOOTSTRAP_REPLICATES):
                sample = _stratified_bootstrap_indices(rows["landcover"], rng)
                auroc_bootstrap.append(roc_auc_score(y[sample], score[sample]))
                balanced_bootstrap.append(
                    balanced_accuracy_score(y[sample], predicted[sample])
                )
            auroc_low, auroc_high = np.quantile(
                auroc_bootstrap, [0.025, 0.975]
            )
            balanced_low, balanced_high = np.quantile(
                balanced_bootstrap, [0.025, 0.975]
            )
            records.append(
                {
                    "task_key": task_key,
                    "window_id": window,
                    "parent_a": parent_a,
                    "parent_b": parent_b,
                    "parent_a_fields": int(y.sum()),
                    "parent_b_fields": int(len(y) - y.sum()),
                    "field_auroc": auroc,
                    "field_auroc_ci_low": float(auroc_low),
                    "field_auroc_ci_high": float(auroc_high),
                    "field_balanced_accuracy_at_0_5": balanced,
                    "field_balanced_accuracy_ci_low": float(balanced_low),
                    "field_balanced_accuracy_ci_high": float(balanced_high),
                    "median_parent_a_score": float(np.median(score[y == 1])),
                    "median_parent_b_score": float(np.median(score[y == 0])),
                }
            )
    return pd.DataFrame(records)


def _gate_table(performance, comparisons, parent_axis_diagnostics):
    records = []
    for task_key, task in TASKS.items():
        hybrid = performance[
            performance["task_key"].eq(task_key)
            & performance["model_key"].eq("hybrid_parent_evidence")
            & performance["stage"].eq(PRIMARY_STAGE)
        ].iloc[0]
        evidence_only = performance[
            performance["task_key"].eq(task_key)
            & performance["model_key"].eq("parent_evidence")
            & performance["stage"].eq(PRIMARY_STAGE)
        ].iloc[0]
        comparison = comparisons[
            comparisons["task_key"].eq(task_key)
            & comparisons["stage"].eq(PRIMARY_STAGE)
            & comparisons["candidate_model_key"].eq("hybrid_parent_evidence")
        ].iloc[0]
        parent_axis = parent_axis_diagnostics[
            parent_axis_diagnostics["task_key"].eq(task_key)
            & parent_axis_diagnostics["window_id"].eq("w3")
        ].iloc[0]
        sample_eligible = int(hybrid["mixture_fields"]) >= MIN_CONFIRMATORY_MIXTURE_FIELDS
        criteria = {
            "sample_eligible": sample_eligible,
            "auroc_at_least_0_75": float(hybrid["auroc"]) >= 0.75,
            "auroc_ci_above_chance": float(hybrid["auroc_ci_low"]) > 0.50,
            "ap_lift_at_least_2": float(hybrid["average_precision_lift"]) >= 2.0,
            "recall_at_least_0_60": float(hybrid["recall"]) >= 0.60,
            "held_out_fpr_at_most_0_15": float(hybrid["false_positive_rate"])
            <= MAX_HELD_OUT_FPR,
            "evidence_only_ap_lift_at_least_1_5": float(
                evidence_only["average_precision_lift"]
            )
            >= 1.5,
            "parent_axis_w3_auroc_at_least_0_70": float(
                parent_axis["field_auroc"]
            )
            >= 0.70,
        }
        mechanism_supported = (
            float(evidence_only["average_precision_lift"]) >= 1.5
            and float(parent_axis["field_auroc"]) >= 0.70
        )
        exploratory_signal = (
            float(hybrid["auroc_ci_low"]) > 0.50
            and float(hybrid["average_precision_lift"]) >= 1.5
            and mechanism_supported
        )
        confirmatory_pass = all(criteria.values())
        if confirmatory_pass:
            status = "PASS"
        elif not sample_eligible and exploratory_signal:
            status = "EXPLORATORY_SAMPLE_TOO_SMALL"
        else:
            status = "FAIL"
        records.append(
            {
                "task_key": task_key,
                "task_title": task["title"],
                "mixture_fields": int(hybrid["mixture_fields"]),
                "status": status,
                "confirmatory_pass": confirmatory_pass,
                "exploratory_signal": exploratory_signal,
                "mechanism_supported": mechanism_supported,
                "auroc": float(hybrid["auroc"]),
                "average_precision": float(hybrid["average_precision"]),
                "average_precision_lift": float(hybrid["average_precision_lift"]),
                "recall": float(hybrid["recall"]),
                "false_positive_rate": float(hybrid["false_positive_rate"]),
                "evidence_only_average_precision_lift": float(
                    evidence_only["average_precision_lift"]
                ),
                "delta_auroc_vs_baseline": float(comparison["delta_auroc"]),
                "delta_ap_vs_baseline": float(
                    comparison["delta_average_precision"]
                ),
                "parent_axis_w3_auroc": float(parent_axis["field_auroc"]),
                "parent_axis_w3_auroc_ci_low": float(
                    parent_axis["field_auroc_ci_low"]
                ),
                **criteria,
            }
        )
    return pd.DataFrame(records)


def run_parent_evidence_evaluation(data_bundle, field_bundle):
    field_table = field_bundle["field_table"]
    all_labels = field_table["landcover"].astype(str).to_numpy()
    all_field_ids = field_table["field_uid"].astype(str).to_numpy()
    prediction_parts = []
    feature_parts = []
    pixel_parts = []
    audit_parts = []
    fold_assignment_parts = []

    for task_offset, (task_key, task) in enumerate(TASKS.items()):
        selected_positions = np.flatnonzero(
            field_table["landcover"].isin(task["classes"]).to_numpy()
        )
        task_labels = all_labels[selected_positions]
        counts = pd.Series(task_labels).value_counts().reindex(task["classes"])
        outer_fold_count = int(min(MAX_OUTER_FOLDS, counts.min()))
        if outer_fold_count < 2:
            raise RuntimeError(f"{task_key} needs at least two fields per class")
        splitter = StratifiedKFold(
            n_splits=outer_fold_count,
            shuffle=True,
            random_state=RANDOM_SEED + task_offset,
        )

        for outer_fold, (train_local, test_local) in enumerate(
            splitter.split(np.zeros(len(task_labels)), task_labels)
        ):
            outer_train = selected_positions[train_local]
            outer_test = selected_positions[test_local]
            train_ids = all_field_ids[outer_train]
            test_ids = all_field_ids[outer_test]
            if set(train_ids) & set(test_ids):
                raise RuntimeError("outer train and test fields overlap")
            train_labels = all_labels[outer_train]
            test_labels = all_labels[outer_test]
            y_train = (train_labels == task["mixture"]).astype(np.int8)
            y_test = (test_labels == task["mixture"]).astype(np.int8)

            train_scores, test_scores, axis_audit = _cross_fitted_parent_scores(
                field_bundle,
                data_bundle["timeline"],
                outer_train,
                outer_test,
                task,
                RANDOM_SEED + 1000 * task_offset + outer_fold,
            )
            axis_audit["task_key"] = task_key
            axis_audit["outer_fold"] = outer_fold
            audit_parts.append(axis_audit)

            train_summary = {
                window: _summarize_parent_scores(train_scores[window], train_ids)
                for window in WINDOWS
            }
            test_summary = {
                window: _summarize_parent_scores(test_scores[window], test_ids)
                for window in WINDOWS
            }
            for window in WINDOWS:
                summary = test_summary[window].reset_index()
                summary["task_key"] = task_key
                summary["outer_fold"] = outer_fold
                summary["window_id"] = window
                summary["is_intercrop"] = summary["landcover"].eq(
                    task["mixture"]
                )
                feature_parts.append(summary)

                pixels = test_scores[window].copy()
                pixels["task_key"] = task_key
                pixels["outer_fold"] = outer_fold
                pixels["window_id"] = window
                pixels["parent_a"] = task["classes"][0]
                pixels["parent_b"] = task["classes"][1]
                pixel_parts.append(pixels)

            assignment = pd.DataFrame(
                {
                    "task_key": task_key,
                    "field_uid": test_ids,
                    "landcover": test_labels,
                    "outer_fold": outer_fold,
                }
            )
            fold_assignment_parts.append(assignment)

            for stage_offset, (stage, stage_windows) in enumerate(STAGES.items()):
                evidence_train, evidence_names = _progressive_parent_matrix(
                    train_summary, stage_windows, train_ids
                )
                evidence_test, test_names = _progressive_parent_matrix(
                    test_summary, stage_windows, test_ids
                )
                if evidence_names != test_names:
                    raise RuntimeError("train/test evidence feature names differ")
                baseline_train = field_bundle["features"][stage][outer_train]
                baseline_test = field_bundle["features"][stage][outer_test]
                model_inputs = {
                    "raw_baseline": (baseline_train, baseline_test),
                    "parent_evidence": (evidence_train, evidence_test),
                    "hybrid_parent_evidence": (
                        np.hstack((baseline_train, evidence_train)),
                        np.hstack((baseline_test, evidence_test)),
                    ),
                }
                for model_offset, (model_key, (x_train, x_test)) in enumerate(
                    model_inputs.items()
                ):
                    fitted = _select_and_fit_detector(
                        x_train,
                        y_train,
                        x_test,
                        RANDOM_SEED
                        + 100000 * task_offset
                        + 1000 * outer_fold
                        + 10 * stage_offset
                        + model_offset,
                    )
                    prediction_parts.append(
                        pd.DataFrame(
                            {
                                "task_key": task_key,
                                "task_title": task["title"],
                                "model_key": model_key,
                                "stage": stage,
                                "outer_fold": outer_fold,
                                "field_uid": test_ids,
                                "landcover": test_labels,
                                "is_intercrop": y_test.astype(bool),
                                "intercrop_score": fitted["scores"],
                                "predicted_intercrop": fitted["predicted"],
                                "threshold": fitted["threshold"],
                                "selected_c": fitted["c_value"],
                                "inner_average_precision": fitted[
                                    "inner_average_precision"
                                ],
                                "inner_auroc": fitted["inner_auroc"],
                                "training_threshold_recall": fitted[
                                    "training_recall"
                                ],
                                "training_threshold_fpr": fitted["training_fpr"],
                            }
                        )
                    )
            print(
                f"completed {task_key} outer fold "
                f"{outer_fold + 1}/{outer_fold_count}"
            )

    predictions = pd.concat(prediction_parts, ignore_index=True)
    features = pd.concat(feature_parts, ignore_index=True)
    pixels = pd.concat(pixel_parts, ignore_index=True)
    axis_audit = pd.concat(audit_parts, ignore_index=True)
    fold_assignments = pd.concat(fold_assignment_parts, ignore_index=True)
    if predictions.duplicated(
        ["task_key", "model_key", "stage", "field_uid"]
    ).any():
        raise RuntimeError("outer OOF predictions contain duplicates")
    if axis_audit["mixture_fields_in_parent_fit"].ne(0).any():
        raise RuntimeError("parent-axis audit detected mixture leakage")
    performance = _performance_table(predictions)
    comparisons = _paired_model_comparison(predictions)
    parent_axis_diagnostics = _parent_axis_diagnostics(features)
    gates = _gate_table(performance, comparisons, parent_axis_diagnostics)
    return {
        "predictions": predictions,
        "performance": performance,
        "comparisons": comparisons,
        "gates": gates,
        "field_window_evidence": features,
        "pixel_parent_evidence": pixels,
        "parent_axis_audit": axis_audit,
        "parent_axis_diagnostics": parent_axis_diagnostics,
        "fold_assignments": fold_assignments,
    }


def figure_v2_design(field_bundle, window_table, diagnostics):
    counts = field_bundle["field_table"]["landcover"].value_counts().reindex(LABELS)
    figure, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    axes[0].barh(
        counts.index,
        counts.to_numpy(),
        color=[CLASS_COLORS[label] for label in counts.index],
    )
    for position, count in enumerate(counts):
        axes[0].text(count, position, f" {count:,}", va="center")
    axes[0].set_title("Frozen physical-field cohort")
    axes[0].set_xlabel("canonical fields")
    axes[0].grid(axis="x", alpha=0.18)

    axes[1].axis("off")
    flow = (
        "OUTER HELD-OUT FIELD FOLD\n\n"
        "1  Fit Bean-vs-Maize or Potato-vs-Maize axes\n"
        "   using monocrop training fields only\n\n"
        "2  Cross-fit pixel parent evidence\n"
        "   and aggregate distribution + spatial features\n\n"
        "3  Select detector regularization and threshold\n"
        f"   inside training data (target FPR <= {TARGET_TRAIN_FPR:.0%})\n\n"
        "4  Score untouched test fields once"
    )
    axes[1].text(
        0.03,
        0.96,
        flow,
        va="top",
        fontsize=11,
        linespacing=1.35,
        bbox={"facecolor": "#F6F8FA", "edgecolor": "#C9D1D9", "pad": 14},
    )
    axes[1].text(
        0.03,
        0.05,
        f"{diagnostics['scoreable_physical_pixels']:,} exact pixels across w1-w4; "
        "w4 remains OOD sensitivity only.",
        fontsize=9,
        color="#4A4F55",
    )
    figure.suptitle("Leakage-safe parent-evidence evaluation", fontsize=16)
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    return figure


def figure_v2_performance(performance):
    metrics = (
        ("auroc", "AUROC"),
        ("average_precision", "Average precision"),
        ("recall", "Intercrop recall"),
        ("false_positive_rate", "Monocrop false-positive rate"),
    )
    stages = list(STAGES)
    x = np.arange(len(stages))
    figure, axes = plt.subplots(2, 4, figsize=(17, 8), sharex=True)
    for row_index, (task_key, task) in enumerate(TASKS.items()):
        for column_index, (metric, title) in enumerate(metrics):
            axis = axes[row_index, column_index]
            for model_key, model_label in MODEL_LABELS.items():
                rows = performance[
                    performance["task_key"].eq(task_key)
                    & performance["model_key"].eq(model_key)
                ].set_index("stage").loc[stages]
                values = rows[metric].to_numpy()
                low = rows[metric + "_ci_low"].to_numpy()
                high = rows[metric + "_ci_high"].to_numpy()
                axis.errorbar(
                    x,
                    values,
                    yerr=np.vstack((values - low, high - values)),
                    marker="o",
                    capsize=2.5,
                    linewidth=2,
                    color=MODEL_COLORS[model_key],
                    label=model_label,
                )
            axis.axvspan(2.5, 3.5, color="#EFEFEF", zorder=-2)
            axis.set_ylim(-0.02, 1.02)
            axis.set_title(title)
            axis.grid(alpha=0.18)
            if column_index == 0:
                axis.set_ylabel(task["title"] + "\nheld-out field score")
            if row_index == 1:
                axis.set_xticks(x, ("w1", "+w2", "+w3", "+w4"), rotation=15)
    axes[0, 3].axhline(MAX_HELD_OUT_FPR, color="#B23A48", linestyle="--")
    axes[1, 3].axhline(MAX_HELD_OUT_FPR, color="#B23A48", linestyle="--")
    axes[0, 0].legend(loc="lower right", fontsize=8)
    figure.suptitle(
        "Does cross-fitted parent evidence improve intercropping detection?",
        fontsize=16,
    )
    figure.tight_layout(rect=(0, 0, 1, 0.95))
    return figure


def figure_v2_precision_recall(predictions):
    figure, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    for axis, (task_key, task) in zip(axes, TASKS.items(), strict=True):
        rows = predictions[
            predictions["task_key"].eq(task_key)
            & predictions["stage"].eq(PRIMARY_STAGE)
        ]
        prevalence = float(rows["is_intercrop"].mean())
        axis.axhline(
            prevalence,
            color="#B23A48",
            linestyle="--",
            linewidth=1.5,
            label=f"prevalence = {prevalence:.1%}",
        )
        for model_key, model_label in MODEL_LABELS.items():
            selected = rows[rows["model_key"].eq(model_key)]
            precision, recall, _ = precision_recall_curve(
                selected["is_intercrop"], selected["intercrop_score"]
            )
            ap = average_precision_score(
                selected["is_intercrop"], selected["intercrop_score"]
            )
            axis.plot(
                recall,
                precision,
                linewidth=2.2,
                color=MODEL_COLORS[model_key],
                label=f"{model_label} (AP={ap:.3f})",
            )
        axis.set_xlim(0, 1)
        axis.set_ylim(0, 1.02)
        axis.set_xlabel("recall")
        axis.set_ylabel("precision")
        axis.set_title(task["title"])
        axis.grid(alpha=0.18)
        axis.legend(fontsize=8)
    figure.suptitle("Held-out field precision-recall after w1+w2+w3", fontsize=16)
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    return figure


def figure_v2_confusions(predictions):
    figure, axes = plt.subplots(1, 2, figsize=(12.5, 5.2))
    for axis, (task_key, task) in zip(axes, TASKS.items(), strict=True):
        rows = predictions[
            predictions["task_key"].eq(task_key)
            & predictions["stage"].eq(PRIMARY_STAGE)
            & predictions["model_key"].eq("hybrid_parent_evidence")
        ]
        matrix = confusion_matrix(
            rows["is_intercrop"],
            rows["predicted_intercrop"],
            labels=[False, True],
            normalize="true",
        )
        image = axis.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
        for row in range(2):
            for column in range(2):
                axis.text(
                    column,
                    row,
                    f"{100 * matrix[row, column]:.0f}%",
                    ha="center",
                    va="center",
                    color="white" if matrix[row, column] > 0.55 else "black",
                    fontsize=13,
                )
        axis.set_xticks((0, 1), ("called monocrop", "called intercrop"))
        axis.set_yticks((0, 1), ("true monocrop", "true intercrop"))
        axis.set_title(task["title"])
        axis.set_xlabel("held-out decision")
    figure.colorbar(image, ax=axes, fraction=0.03, pad=0.05, label="within-class rate")
    figure.suptitle(
        "Hybrid parent-evidence decisions at the predeclared operating rule",
        fontsize=15,
    )
    figure.subplots_adjust(top=0.84, bottom=0.16, wspace=0.32)
    return figure


def figure_v2_score_distributions(predictions):
    figure, axes = plt.subplots(1, 2, figsize=(14, 5.6), sharey=True)
    for axis, (task_key, task) in zip(axes, TASKS.items(), strict=True):
        rows = predictions[
            predictions["task_key"].eq(task_key)
            & predictions["stage"].eq(PRIMARY_STAGE)
            & predictions["model_key"].eq("hybrid_parent_evidence")
        ]
        values = [
            rows.loc[rows["landcover"].eq(label), "intercrop_score"].to_numpy()
            for label in task["classes"]
        ]
        boxes = axis.boxplot(
            values,
            patch_artist=True,
            showfliers=True,
        )
        axis.set_xticks(
            np.arange(1, len(task["classes"]) + 1),
            [label.replace(" and ", "+") for label in task["classes"]],
        )
        for patch, label in zip(boxes["boxes"], task["classes"], strict=True):
            patch.set_facecolor(CLASS_COLORS[label])
            patch.set_alpha(0.65)
        axis.set_title(task["title"])
        axis.set_ylabel("held-out hybrid intercropping score")
        axis.set_ylim(-0.02, 1.02)
        axis.tick_params(axis="x", rotation=18)
        axis.grid(axis="y", alpha=0.18)
    figure.suptitle("Field-level score separation after w1+w2+w3", fontsize=16)
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    return figure


def figure_v2_parent_trajectories(field_evidence):
    figure, axes = plt.subplots(2, 3, figsize=(17, 9), sharex=True)
    metrics = (
        ("mean_parent_a", "Mean pixel evidence for parent A"),
        ("parent_balance", "Parent balance (1 = centered)"),
        ("q90_q10_parent_a", "Within-field parent-evidence spread"),
    )
    x = np.arange(len(PRIMARY_WINDOWS))
    for row_index, (task_key, task) in enumerate(TASKS.items()):
        task_rows = field_evidence[field_evidence["task_key"].eq(task_key)]
        for column_index, (metric, title) in enumerate(metrics):
            axis = axes[row_index, column_index]
            for label in task["classes"]:
                selected = task_rows[task_rows["landcover"].eq(label)]
                grouped = selected.groupby("window_id")[metric]
                median = grouped.median().reindex(PRIMARY_WINDOWS)
                low = grouped.quantile(0.25).reindex(PRIMARY_WINDOWS)
                high = grouped.quantile(0.75).reindex(PRIMARY_WINDOWS)
                color = CLASS_COLORS[label]
                axis.fill_between(x, low, high, color=color, alpha=0.14)
                axis.plot(x, median, marker="o", linewidth=2, color=color, label=label)
            axis.set_ylim(-0.02, 1.02)
            axis.set_title(title)
            axis.grid(alpha=0.18)
            if column_index == 0:
                axis.set_ylabel(task["title"])
            if row_index == 1:
                axis.set_xticks(x, PRIMARY_WINDOWS)
            axis.legend(fontsize=7.5)
    figure.suptitle(
        "How cross-fitted parent evidence evolves through the in-contract windows",
        fontsize=16,
    )
    figure.tight_layout(rect=(0, 0, 1, 0.95))
    return figure


def _iter_polygons_v2(geometry):
    if geometry.geom_type == "Polygon":
        yield geometry
    elif geometry.geom_type in {"MultiPolygon", "GeometryCollection"}:
        for child in geometry.geoms:
            yield from _iter_polygons_v2(child)


def _draw_outline_v2(axis, geometry):
    for polygon in _iter_polygons_v2(geometry):
        x, y = polygon.exterior.xy
        axis.plot(x, y, color="black", linewidth=1.2, zorder=4)


def _representative_v2_fields(results):
    records = []
    rows = results["predictions"]
    for task_key, task in TASKS.items():
        candidates = rows[
            rows["task_key"].eq(task_key)
            & rows["stage"].eq(PRIMARY_STAGE)
            & rows["model_key"].eq("hybrid_parent_evidence")
            & rows["is_intercrop"]
        ].copy()
        true_positive = candidates[candidates["predicted_intercrop"]]
        pool = true_positive if not true_positive.empty else candidates
        target = float(pool["intercrop_score"].median())
        selected = pool.assign(
            _distance=(pool["intercrop_score"] - target).abs()
        ).sort_values(["_distance", "field_uid"]).iloc[0]
        records.append(
            {
                "task_key": task_key,
                "field_uid": str(selected["field_uid"]),
                "landcover": task["mixture"],
                "selection_basis": (
                    "median-score true-positive held-out field"
                    if not true_positive.empty
                    else "median-score held-out mixture; no true positives"
                ),
                "primary_intercrop_score": float(selected["intercrop_score"]),
            }
        )
    return pd.DataFrame(records)


def figure_v2_pixel_maps(results, field_bundle):
    selections = _representative_v2_fields(results)
    figure, axes = plt.subplots(2, 3, figsize=(15, 9))
    field_table = field_bundle["field_table"].set_index("field_uid")
    collection = None
    for row_index, selection in enumerate(selections.itertuples(index=False)):
        task = TASKS[selection.task_key]
        field_row = field_table.loc[selection.field_uid]
        field_pixels = results["pixel_parent_evidence"][
            results["pixel_parent_evidence"]["task_key"].eq(selection.task_key)
            & results["pixel_parent_evidence"]["field_uid"].eq(selection.field_uid)
            & results["pixel_parent_evidence"]["window_id"].eq("w3")
        ]
        epsg = int(field_pixels["utm_epsg"].iloc[0])
        transformer = Transformer.from_crs(4326, epsg, always_xy=True)
        geometry = shapely_transform(
            transformer.transform,
            shapely_wkt.loads(str(field_row["wkt"])),
        )
        left = field_pixels["pixel_x_index"].to_numpy() * PIXEL_SIZE_M
        bottom = field_pixels["pixel_y_index"].to_numpy() * PIXEL_SIZE_M
        min_x, min_y, max_x, max_y = geometry.bounds
        min_x = min(min_x, float(left.min()))
        min_y = min(min_y, float(bottom.min()))
        max_x = max(max_x, float((left + PIXEL_SIZE_M).max()))
        max_y = max(max_y, float((bottom + PIXEL_SIZE_M).max()))
        pad = max(PIXEL_SIZE_M, 0.05 * max(max_x - min_x, max_y - min_y))

        for column_index, window in enumerate(PRIMARY_WINDOWS):
            axis = axes[row_index, column_index]
            pixels = results["pixel_parent_evidence"][
                results["pixel_parent_evidence"]["task_key"].eq(selection.task_key)
                & results["pixel_parent_evidence"]["field_uid"].eq(selection.field_uid)
                & results["pixel_parent_evidence"]["window_id"].eq(window)
            ]
            rectangles = [
                Rectangle(
                    (
                        float(row.pixel_x_index) * PIXEL_SIZE_M,
                        float(row.pixel_y_index) * PIXEL_SIZE_M,
                    ),
                    PIXEL_SIZE_M,
                    PIXEL_SIZE_M,
                )
                for row in pixels.itertuples(index=False)
            ]
            collection = PatchCollection(
                rectangles,
                cmap="BrBG",
                norm=Normalize(0, 1),
                edgecolor="white",
                linewidth=0.35,
            )
            collection.set_array(pixels["parent_a_probability"].to_numpy())
            axis.add_collection(collection)
            _draw_outline_v2(axis, geometry)
            stage = list(STAGES)[column_index]
            field_score = results["predictions"].loc[
                results["predictions"]["task_key"].eq(selection.task_key)
                & results["predictions"]["field_uid"].eq(selection.field_uid)
                & results["predictions"]["model_key"].eq(
                    "hybrid_parent_evidence"
                )
                & results["predictions"]["stage"].eq(stage),
                "intercrop_score",
            ].iloc[0]
            axis.set_title(
                f"{window} | field intercrop score={field_score:.2f}\n"
                f"mean {task['classes'][0]} evidence={pixels['parent_a_probability'].mean():.2f}"
            )
            axis.set_xlim(min_x - pad, max_x + pad)
            axis.set_ylim(min_y - pad, max_y + pad)
            axis.set_aspect("equal")
            axis.set_xticks([])
            axis.set_yticks([])
        axes[row_index, 0].set_ylabel(
            f"{selection.landcover}\n{selection.field_uid}", fontsize=9
        )
    if collection is not None:
        colorbar = figure.colorbar(
            collection,
            ax=axes,
            orientation="horizontal",
            fraction=0.035,
            pad=0.07,
        )
        colorbar.set_label("pixel parent evidence: 0 = Maize, 1 = other named parent")
    figure.suptitle(
        "Representative held-out fields: spatial parent evidence through time",
        fontsize=15,
    )
    figure.subplots_adjust(top=0.88, bottom=0.13, hspace=0.28, wspace=0.16)
    return figure, selections


def build_v2_handoff(data_bundle, field_bundle, results, representative_fields):
    performance_columns = [
        "task_key",
        "task_title",
        "model_key",
        "model",
        "stage",
        "contract_status",
        "fields",
        "mixture_fields",
        "prevalence",
        "auroc",
        "auroc_ci_low",
        "auroc_ci_high",
        "average_precision",
        "average_precision_ci_low",
        "average_precision_ci_high",
        "average_precision_lift",
        "recall",
        "recall_ci_low",
        "recall_ci_high",
        "false_positive_rate",
        "false_positive_rate_ci_low",
        "false_positive_rate_ci_high",
        "precision",
        "balanced_accuracy",
    ]
    counts = (
        field_bundle["field_table"].groupby("landcover", as_index=False)
        .agg(physical_fields=("field_uid", "nunique"))
        .sort_values("physical_fields", ascending=False)
    )
    primary_predictions = results["predictions"][
        results["predictions"]["stage"].eq(PRIMARY_STAGE)
    ][
        [
            "task_key",
            "model_key",
            "field_uid",
            "landcover",
            "is_intercrop",
            "intercrop_score",
            "predicted_intercrop",
        ]
    ]
    trajectory = (
        results["field_window_evidence"]
        .groupby(["task_key", "window_id", "landcover"], as_index=False)
        .agg(
            fields=("field_uid", "nunique"),
            mean_parent_a_q25=("mean_parent_a", lambda values: values.quantile(0.25)),
            mean_parent_a_median=("mean_parent_a", "median"),
            mean_parent_a_q75=("mean_parent_a", lambda values: values.quantile(0.75)),
            dual_tail_q25=("dual_parent_tail_mass", lambda values: values.quantile(0.25)),
            dual_tail_median=("dual_parent_tail_mass", "median"),
            dual_tail_q75=("dual_parent_tail_mass", lambda values: values.quantile(0.75)),
            parent_balance_q25=("parent_balance", lambda values: values.quantile(0.25)),
            parent_balance_median=("parent_balance", "median"),
            parent_balance_q75=("parent_balance", lambda values: values.quantile(0.75)),
            parent_spread_q25=("q90_q10_parent_a", lambda values: values.quantile(0.25)),
            parent_spread_median=("q90_q10_parent_a", "median"),
            parent_spread_q75=("q90_q10_parent_a", lambda values: values.quantile(0.75)),
        )
    )
    indexed_fields = field_bundle["field_table"].set_index("field_uid")
    representative_maps = []
    for selection in representative_fields.itertuples(index=False):
        task = TASKS[selection.task_key]
        field_row = indexed_fields.loc[selection.field_uid]
        stage_records = []
        for window, stage in zip(
            PRIMARY_WINDOWS,
            list(STAGES)[: len(PRIMARY_WINDOWS)],
            strict=True,
        ):
            pixels = results["pixel_parent_evidence"][
                results["pixel_parent_evidence"]["task_key"].eq(selection.task_key)
                & results["pixel_parent_evidence"]["field_uid"].eq(selection.field_uid)
                & results["pixel_parent_evidence"]["window_id"].eq(window)
            ][
                [
                    "pixel_id",
                    "utm_epsg",
                    "pixel_x_index",
                    "pixel_y_index",
                    "parent_a_probability",
                ]
            ]
            field_score = results["predictions"].loc[
                results["predictions"]["task_key"].eq(selection.task_key)
                & results["predictions"]["field_uid"].eq(selection.field_uid)
                & results["predictions"]["model_key"].eq(
                    "hybrid_parent_evidence"
                )
                & results["predictions"]["stage"].eq(stage),
                "intercrop_score",
            ].iloc[0]
            stage_records.append(
                {
                    "window_id": window,
                    "stage": stage,
                    "field_intercrop_score": float(field_score),
                    "pixels": json.loads(
                        pixels.to_json(orient="records", double_precision=5)
                    ),
                }
            )
        representative_maps.append(
            {
                "task_key": selection.task_key,
                "field_uid": selection.field_uid,
                "landcover": selection.landcover,
                "parent_a": task["classes"][0],
                "parent_b": task["classes"][1],
                "wkt": str(field_row["wkt"]),
                "stages": stage_records,
            }
        )
    return {
        "schema": "spectrajam.intercropping_parent_evidence.v2",
        "analysis_question": (
            "Can cross-fitted monocrop parent signatures detect intercropped fields "
            "at a controlled false-positive operating point?"
        ),
        "diagnostics": data_bundle["diagnostics"],
        "windows": json.loads(data_bundle["window_table"].to_json(orient="records")),
        "field_counts": json.loads(counts.to_json(orient="records")),
        "validation_contract": {
            "outer_unit": "canonical physical field",
            "parent_axis_training": "monocrop outer-training fields only",
            "parent_evidence": "cross-fitted within outer training; fold ensemble on outer test",
            "detector_selection": "inner field CV maximizing average precision",
            "threshold_rule": f"training FPR <= {TARGET_TRAIN_FPR:.2f}, then maximize recall",
            "primary_candidate": "raw + cross-fitted parent-evidence hybrid",
            "mechanism_check": "parent-evidence-only average-precision lift",
            "primary_stage": PRIMARY_STAGE,
            "w4_policy": "OOD sensitivity only",
            "spatial_transfer_claim": False,
        },
        "success_gate": {
            "minimum_mixture_fields": MIN_CONFIRMATORY_MIXTURE_FIELDS,
            "minimum_auroc": 0.75,
            "auroc_ci_low_must_exceed": 0.50,
            "minimum_average_precision_lift": 2.0,
            "minimum_recall": 0.60,
            "maximum_held_out_fpr": MAX_HELD_OUT_FPR,
            "minimum_w3_parent_axis_auroc": 0.70,
        },
        "performance": json.loads(
            results["performance"][performance_columns].to_json(
                orient="records", double_precision=5
            )
        ),
        "paired_comparisons": json.loads(
            results["comparisons"].to_json(orient="records", double_precision=5)
        ),
        "gates": json.loads(results["gates"].to_json(orient="records")),
        "parent_axis_diagnostics": json.loads(
            results["parent_axis_diagnostics"].to_json(
                orient="records", double_precision=5
            )
        ),
        "primary_predictions": json.loads(
            primary_predictions.to_json(orient="records", double_precision=5)
        ),
        "parent_evidence_trajectories": json.loads(
            trajectory.to_json(orient="records", double_precision=5)
        ),
        "representative_fields": json.loads(
            representative_fields.to_json(orient="records", double_precision=5)
        ),
        "representative_pixel_maps": representative_maps,
        "interpretation_boundary": (
            "Parent evidence is relative TESSERA embedding affinity. It is not "
            "planted-area share, biomass, plant count, yield, or genetic ancestry."
        ),
    }


## 1. Freeze the same longitudinal physical-field cohort

The analysis keeps one canonical unit per same-label geometry, excludes conflicting-label geometries, removes pixels shared by different physical fields, and requires the exact same finite pixels in all four windows.


In [ ]:
data_bundle = load_longitudinal_embeddings(OUTPUT_DIR)
field_bundle = build_field_features(data_bundle["timeline"], data_bundle["fields"])
export_root = PARENT_EVIDENCE_EXPORT_BASE / data_bundle["snapshot_id"]
export_root.mkdir(parents=True, exist_ok=True)
(export_root / "COMPLETED.json").unlink(missing_ok=True)

print("Input:", OUTPUT_DIR)
print("Frozen export:", export_root)
print(json.dumps(data_bundle["diagnostics"], indent=2))
display(data_bundle["window_table"])
display(
    field_bundle["field_table"]
    .groupby("landcover")
    .agg(physical_fields=("field_uid", "nunique"))
    .sort_values("physical_fields", ascending=False)
)


## 2. Predeclared validation and success contract

The operating threshold targets at most 10% false positives inside each outer training fold and then maximizes recall. The reported false-positive rate is always the observed rate on untouched outer fields. A confirmatory pass requires enough intercropped fields, AUROC and its uncertainty above chance, useful average-precision lift, at least 60% recall, and no more than 15% held-out monocrop false positives.


In [ ]:
validation_contract = pd.DataFrame(
    [
        ("Outer evaluation unit", "canonical physical field"),
        ("Parent signature training", "field-balanced outer-training monocrop pixels only"),
        ("Parent feature construction", "cross-fitted; fold ensemble on outer test"),
        ("Detector model selection", "inner field CV, maximum average precision"),
        ("Threshold rule", "training FPR <= 10%, then maximum recall"),
        ("Primary candidate", "raw + cross-fitted parent-evidence hybrid"),
        ("Mechanism check", "parent-evidence-only average-precision lift"),
        ("Primary endpoint", PRIMARY_STAGE),
        ("Minimum confirmatory intercrop fields", MIN_CONFIRMATORY_MIXTURE_FIELDS),
        ("w4", "OOD sensitivity only"),
    ],
    columns=["component", "contract"],
)
display(validation_contract)


## 3. Run the nested held-out-field evaluation

The raw mean-plus-standard-deviation detector is retained as a matched binary baseline. Parent evidence is also tested alone. The primary candidate is the hybrid raw-plus-parent representation, which avoids assuming that every intercropping signal must reduce to a two-parent mixture. All three models receive identical outer folds and identical threshold-selection rules. Pixel rows are never counted as independent validation samples.


In [ ]:
results = run_parent_evidence_evaluation(data_bundle, field_bundle)

tables = export_root / "tables"
_atomic_parquet(results["predictions"], tables / "field_oof_predictions.parquet")
_atomic_parquet(results["performance"], tables / "field_performance.parquet")
_atomic_parquet(results["comparisons"], tables / "paired_model_comparisons.parquet")
_atomic_parquet(results["gates"], tables / "success_gates.parquet")
_atomic_parquet(
    results["field_window_evidence"],
    tables / "field_window_parent_evidence.parquet",
)
_atomic_parquet(
    results["pixel_parent_evidence"],
    tables / "pixel_oof_parent_evidence.parquet",
)
_atomic_parquet(results["parent_axis_audit"], tables / "parent_axis_audit.parquet")
_atomic_parquet(
    results["parent_axis_diagnostics"],
    tables / "parent_axis_diagnostics.parquet",
)
_atomic_parquet(results["fold_assignments"], tables / "outer_fold_assignments.parquet")

display(
    results["performance"][
        [
            "task_title",
            "model",
            "stage",
            "fields",
            "mixture_fields",
            "auroc",
            "average_precision",
            "average_precision_lift",
            "recall",
            "false_positive_rate",
            "precision",
        ]
    ].round(3)
)
display(results["gates"])
display(results["parent_axis_diagnostics"].round(3))


## Figure 1. Cohort and leakage controls

This is the denominator and evaluation-flow figure. It makes explicit that parent signatures and detector thresholds are learned without access to the outer test fields.


In [ ]:
figure_1 = figure_v2_design(
    field_bundle,
    data_bundle["window_table"],
    data_bundle["diagnostics"],
)
figure_1_path = _save_figure(figure_1, export_root, "01_design_and_cohort.png")
display(figure_1)
plt.close(figure_1)
print(figure_1_path)


## Figure 2. Parent evidence versus the raw baseline

This is the primary model-comparison figure. AUROC and average precision measure ranking; recall and false-positive rate show the actual fold-specific operating decisions. Error bars are stratified physical-field bootstrap 95% intervals. The hybrid is the predeclared primary candidate.


In [ ]:
figure_2 = figure_v2_performance(results["performance"])
figure_2_path = _save_figure(figure_2, export_root, "02_model_comparison.png")
display(figure_2)
plt.close(figure_2)
print(figure_2_path)


## Figure 3. Precision-recall at the primary endpoint

Average precision must be read against each family's intercropping prevalence. This is especially important for Potato-Maize, where only 11 of 386 fields are intercropped.


In [ ]:
figure_3 = figure_v2_precision_recall(results["predictions"])
figure_3_path = _save_figure(figure_3, export_root, "03_primary_precision_recall.png")
display(figure_3)
plt.close(figure_3)
print(figure_3_path)


## Figure 4. Held-out decisions at the operating rule

Rows are true binary field states and columns are held-out hybrid-model calls. These are not resubstitution or pixel-level counts.


In [ ]:
figure_4 = figure_v2_confusions(results["predictions"])
figure_4_path = _save_figure(figure_4, export_root, "04_primary_binary_confusions.png")
display(figure_4)
plt.close(figure_4)
print(figure_4_path)


## Figure 5. Field-level score separation

This shows the complete held-out hybrid-score distributions, rather than selected examples. A useful detector should shift the intercropping distribution upward while keeping both monocrop distributions low.


In [ ]:
figure_5 = figure_v2_score_distributions(results["predictions"])
figure_5_path = _save_figure(figure_5, export_root, "05_primary_score_distributions.png")
display(figure_5)
plt.close(figure_5)
print(figure_5_path)


## Figure 6. Parent-signature evolution

The first column tracks which monocrop parent the pixels resemble on average. The second tracks whether the field-average evidence is balanced between the parents. The third tracks within-field spread along the parent axis. Medians and interquartile ranges are computed from outer-held-out field features.


In [ ]:
figure_6 = figure_v2_parent_trajectories(results["field_window_evidence"])
figure_6_path = _save_figure(figure_6, export_root, "06_parent_evidence_trajectories.png")
display(figure_6)
plt.close(figure_6)
print(figure_6_path)


## Figure 7. Spatial parent evidence inside representative held-out fields

Each row is a deterministic median-score true positive when available. The color shows the parent axis, not intercropped area percentage. This is an explanatory case study; the field-level outer-fold metrics remain the evidence.


In [ ]:
figure_7, representative_fields = figure_v2_pixel_maps(results, field_bundle)
figure_7_path = _save_figure(figure_7, export_root, "07_representative_parent_maps.png")
display(figure_7)
plt.close(figure_7)
display(representative_fields)
print(figure_7_path)


## Completion audit and one-block handoff

The final block contains the validation contract, every primary held-out score, performance intervals, paired comparisons, gate decisions, parent-evidence trajectories, and representative-field metadata. Copy only this one block when the run is complete.


In [ ]:
handoff = build_v2_handoff(
    data_bundle,
    field_bundle,
    results,
    representative_fields,
)
figure_paths = [
    figure_1_path,
    figure_2_path,
    figure_3_path,
    figure_4_path,
    figure_5_path,
    figure_6_path,
    figure_7_path,
]
manifest = {
    "analysis_name": "intercropping_parent_evidence_v2",
    "snapshot_id": data_bundle["snapshot_id"],
    "figures": [
        {
            "path": str(path.relative_to(export_root)),
            "bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
        for path in figure_paths
    ],
    "tables": [
        str(path.relative_to(export_root))
        for path in sorted((export_root / "tables").glob("*.parquet"))
    ],
}
handoff["figure_manifest"] = manifest["figures"]
_atomic_json(handoff, export_root / "PDF_HANDOFF_V2.json")
_atomic_json(manifest, export_root / "manifest.json")
_atomic_json(
    {
        "analysis_name": manifest["analysis_name"],
        "snapshot_id": data_bundle["snapshot_id"],
        "figure_count": len(figure_paths),
        "gate_statuses": dict(
            results["gates"][["task_key", "status"]].itertuples(index=False, name=None)
        ),
        "complete": True,
    },
    export_root / "COMPLETED.json",
)

print("PDF_HANDOFF_V2_BEGIN")
print(json.dumps(handoff, separators=(",", ":"), allow_nan=False))
print("PDF_HANDOFF_V2_END")
